# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Duchalsoham12/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [9]:
# Create ranked action queue

queue = test[
    [
        "content_id",
        "content_age_days",
        "days_since_last_update",
        "engagement_rate",
        "ai_traffic_pct",
        "trend_direction",
        "trend_pct"
    ]
].copy()

queue["decline_score"] = pred_prob

# Reason code
def reason_code(row):
    reasons = []

    if row["days_since_last_update"] > 180:
        reasons.append("STALE_CONTENT")

    if pd.notna(row["engagement_rate"]) and row["engagement_rate"] < 0.50:
        reasons.append("LOW_ENGAGEMENT")

    if pd.notna(row["trend_pct"]) and row["trend_pct"] < 0:
        reasons.append("NEGATIVE_TREND")

    if not reasons:
        reasons.append("MODEL_PRIORITY")

    return "|".join(reasons)

queue["reason_code"] = queue.apply(reason_code, axis=1)

# Highest priority first
queue = queue.sort_values(
    "decline_score",
    ascending=False
).reset_index(drop=True)

queue["priority_rank"] = np.arange(1, len(queue) + 1)

display(queue.head(20))

,content_id,content_age_days,days_since_last_update,engagement_rate,ai_traffic_pct,trend_direction,trend_pct,decline_score,reason_code,priority_rank
0,content_29884c0f9255,223,102,0.00,0.00,down,-78.0,0.976667,LOW_ENGAGEMENT|NEGATIVE_TREND,1
1,content_eb3b2c3bbc34,117,20,0.00,0.00,down,-96.2,0.970000,LOW_ENGAGEMENT|NEGATIVE_TREND,2
2,content_41538bdb1b1e,97,8,0.00,0.00,down,-98.6,0.966667,LOW_ENGAGEMENT|NEGATIVE_TREND,3
3,content_9234f5075e7a,95,20,0.00,0.00,down,-93.0,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,4
4,content_9ac61c04930e,275,104,0.00,0.00,down,-53.9,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,5
5,content_f6bf66378677,141,20,0.00,0.00,down,-86.1,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,6
6,content_9e8671965fff,95,20,0.00,0.00,down,-82.4,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND,7
7,content_c6bb205d5263,95,20,0.00,0.00,down,-49.1,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND,8
8,content_8b08ec7fc725,237,103,0.00,0.00,down,-100.0,0.953333,LOW_ENGAGEMENT|NEGATIVE_TREND,9
9,content_e5fd30b6e33b,141,20,0.00,0.00,down,-68.0,0.950000,LOW_ENGAGEMENT|NEGATIVE_TREND,10


In [10]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# --------------------------------------------------
# 1. Find and load dataset
# --------------------------------------------------

repo_path = "flyrank-ml-internship"

if not os.path.exists(repo_path):
    !git clone https://github.com/Duchalsoham12/flyrank-ml-internship.git

csv_path = None

for root, dirs, files in os.walk(repo_path):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            csv_path = os.path.join(root, file)

if csv_path is None:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv not found."
    )

df = pd.read_csv(csv_path)

# Target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Dataset:", df.shape)

# --------------------------------------------------
# 2. Client-grouped split
# --------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train:", train.shape)
print("Test:", test.shape)

# --------------------------------------------------
# 3. Prepare features
# --------------------------------------------------

excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

features = [
    c for c in df.columns
    if c not in excluded
]

X_train = train[features]
y_train = train["is_declining_label"]

X_test = test[features]
y_test = test["is_declining_label"]

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

# --------------------------------------------------
# 4. Train Random Forest
# --------------------------------------------------

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

# --------------------------------------------------
# 5. Predictions
# --------------------------------------------------

pred_prob = model.predict_proba(X_test)[:, 1]

# Precision@50
top_n = min(50, len(y_test))
top_indices = np.argsort(pred_prob)[::-1][:top_n]

honest_p50 = y_test.iloc[top_indices].mean()

print("Model trained successfully.")
print("Precision@50:", round(honest_p50, 3))
print("Client overlap:",
      len(set(train["client_id"]) & set(test["client_id"])))

Dataset: (30000, 45)
Train: (23837, 45)
Test: (6163, 45)
Model trained successfully.
Precision@50: 1.0
Client overlap: 0


### 1. Ranked actions + reason codes

The queue ranks content using the model's measured probability of decline. Higher scores are reviewed first. Reason codes add simple signals that help a human understand why an item was prioritized. The queue is for decision-support, not automatic action.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [11]:
# Check the ranked queue

print("Queue size:", len(queue))

print(
    "Highest decline score:",
    round(queue["decline_score"].max(), 3)
)

print(
    "Lowest decline score:",
    round(queue["decline_score"].min(), 3)
)

print("\nReason-code counts:")

display(
    queue["reason_code"]
    .value_counts()
    .head(10)
    .to_frame("count")
)

Queue size: 6163
Highest decline score: 0.977
Lowest decline score: 0.003

Reason-code counts:


,count
reason_code,
LOW_ENGAGEMENT|NEGATIVE_TREND,2995
LOW_ENGAGEMENT,1829
NEGATIVE_TREND,874
MODEL_PRIORITY,465


### 2. Intended use and limits

The queue is intended to help a content team prioritize pages for human review. Higher model scores indicate higher measured decline risk in this dataset. The model should not be used to automatically delete, rewrite, redirect, or publish content. Its results are directional decision-support and may become less reliable when the data or content environment changes.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [12]:
# Create a small human-review queue

review_queue = queue.head(20).copy()

print("Items requiring human review:", len(review_queue))

display(
    review_queue[
        [
            "priority_rank",
            "content_id",
            "decline_score",
            "reason_code"
        ]
    ]
)

Items requiring human review: 20


,priority_rank,content_id,decline_score,reason_code
0,1,content_29884c0f9255,0.976667,LOW_ENGAGEMENT|NEGATIVE_TREND
1,2,content_eb3b2c3bbc34,0.970000,LOW_ENGAGEMENT|NEGATIVE_TREND
2,3,content_41538bdb1b1e,0.966667,LOW_ENGAGEMENT|NEGATIVE_TREND
3,4,content_9234f5075e7a,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND
4,5,content_9ac61c04930e,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND
5,6,content_f6bf66378677,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND
6,7,content_9e8671965fff,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND
7,8,content_c6bb205d5263,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND
8,9,content_8b08ec7fc725,0.953333,LOW_ENGAGEMENT|NEGATIVE_TREND
9,10,content_e5fd30b6e33b,0.950000,LOW_ENGAGEMENT|NEGATIVE_TREND


### 3. Human review + the no-go list

A person should review the page's search intent, factual accuracy, content quality, business relevance, and the reason for the observed decline before taking action. The model score is only a prioritization signal.

**No-go list:** Do not automatically delete content, publish rewritten content, change factual claims, remove important pages, or make irreversible SEO changes based only on the model score.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [13]:
# Current monitoring baseline

monitoring_df = pd.DataFrame({
    "metric": [
        "Test rows",
        "Decline rate",
        "Precision@50",
        "Test clients"
    ],
    "value": [
        len(test),
        round(y_test.mean(), 3),
        round(honest_p50, 3),
        test["client_id"].nunique()
    ]
})

display(monitoring_df)

,metric,value
0,Test rows,6163.000
1,Decline rate,0.511
2,Precision@50,1.000
3,Test clients,7.000


### 4. Monitoring / retrain triggers

The recommendations should be reviewed if Precision@50 falls, the observed decline rate changes substantially, or important feature distributions shift. A retrain should be considered when new labeled data is available and the current model no longer provides useful directional decision-support.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [14]:
import os

# Create the required output folder
output_dir = "flyrank-ml-internship/work/outputs"
os.makedirs(output_dir, exist_ok=True)

# Save ranked action queue
queue_path = os.path.join(
    output_dir,
    "ml10_ranked_action_queue.csv"
)

queue.to_csv(
    queue_path,
    index=False
)

# Save monitoring results
monitoring_path = os.path.join(
    output_dir,
    "ml10_monitoring_summary.csv"
)

monitoring_df.to_csv(
    monitoring_path,
    index=False
)

print("Files exported successfully:")
print(queue_path)
print(monitoring_path)

Files exported successfully:
flyrank-ml-internship/work/outputs/ml10_ranked_action_queue.csv
flyrank-ml-internship/work/outputs/ml10_monitoring_summary.csv


In [15]:
print("ML-10 completed successfully.")

print("\nQueue rows:", len(queue))
print("Precision@50:", round(honest_p50, 3))

print("\nOutput files:")
for file in os.listdir(output_dir):
    print("-", file)

ML-10 completed successfully.

Queue rows: 6163
Precision@50: 1.0

Output files:
- ml10_monitoring_summary.csv
- ml10_ranked_action_queue.csv


### 5. Exports for the paper

The ranked action queue and monitoring summary are exported to the required outputs folder. These files provide measured decision-support results that can be reused in the paper.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.